In [ ]:
import math
import os
import numpy as np
import pandas as pd
from scipy.stats import entropy
from collections import Counter
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Como o notebook está em 'corretor/src/3_avaliacao' e o .env em 'corretor',
# precisamos voltar duas pastas (../..) para achar o arquivo.
env_path = Path('../../.env')

# Carrega o arquivo explicitamente
load_dotenv(dotenv_path=env_path)

In [ ]:
PATH_BASE_ANONIMIZADA_CORRETOR = os.getenv("PATH_BASE_ANONIMIZADA_CORRETOR")
PATH_BASE_ENTROPIA_CORRETOR = os.getenv("PATH_BASE_ENTROPIA_CORRETOR")

In [ ]:
base_relato_anon = pd.read_csv(PATH_BASE_ANONIMIZADA_CORRETOR)

É realizado o cálculo de entropia dos textos em claro e anonimizados, após isso realizada a comparação dos resultados obtidos.

- Foi criada uma tabela com o resultado da entropia de cada texto e realizada
- Realizado o cálculo da média de cada conjunto: Média dos resultados dos textos em claro e média dos resultados dos textos anonimizados. Assim podendo quantificar o impacto.
- Também foi realizado o cálculo do desvio padrão para mensurar diferença das entropias.

A entropia é calulada com base na probabilidade da ocorrência de cada caracter no texto. O resultado obtido da probabilidade de cada caracter é inserido para calculo da entropia utilizando a biblioteca entropy do scipy.stats que calcula a entropia com base na formula da entropia de shannon ( entropia da informação ).


In [ ]:
def calculate_text_entropy(text, base=None):
    char_counts = Counter(text)
    total_chars = sum(char_counts.values())
    probabilities = np.array([count / total_chars for count in char_counts.values()])

    return entropy(probabilities, base=base)

resultado texto unicos

In [ ]:
text = base_relato_anon['relato'].iloc[0]
text_anon = base_relato_anon['relato_anon'].iloc[0]

entropia do texto original

In [ ]:
entropia = calculate_text_entropy(text, base=2)
entropia

entropia do texto anonimizado

In [ ]:
entropia = calculate_text_entropy(text_anon, base=2)
entropia

resultado sobre toda a base de BOs

In [ ]:
base_relato_anon['entrpy_result_orig'] = base_relato_anon['relato'].apply(lambda x: calculate_text_entropy(x, base=2))

In [ ]:
base_relato_anon['entrpy_result_anon'] = base_relato_anon['relato_anon'].apply(lambda x: calculate_text_entropy(x, base=2))

In [ ]:
save_df = base_relato_anon[["n_bop", "entrpy_result_orig", "entrpy_result_anon"]]

In [ ]:
save_df = save_df.rename(columns={"entrpy_result_orig":"corretor_text_orig", "entrpy_result_anon":"Entropy Corretor"})

In [ ]:
save_df.to_csv(PATH_BASE_ENTROPIA_CORRETOR, index=False)

In [ ]:
save_df

### Verificando se há outliers

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(base_relato_anon['entrpy_result_orig'], patch_artist=True)
plt.grid(True)
plt.title('Validação Cruzada')
plt.ylabel('Acurácia')
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(base_relato_anon['entrpy_result_anon'], patch_artist=True)
plt.grid(True)
plt.title('Validação Cruzada')
plt.ylabel('Acurácia')
plt.show()


#### Teste Shapiro para verificação da distribuição

In [ ]:
shapiro_orig = stats.shapiro(base_relato_anon["entrpy_result_orig"])
shapiro_anon = stats.shapiro(base_relato_anon["entrpy_result_anon"])
print(f"Teste de Shapiro-Wilk para entrpy_result_orig: p-valor={shapiro_orig.pvalue}")
print(f"Teste de Shapiro-Wilk para entrpy_result_anon: p-valor={shapiro_anon.pvalue}")

#### Teste de Wilcoxon - verificação da significancia

recomendação para utilização desse teste em casos que houvesse outliers - o que foi visto nas células acima

In [ ]:
w_stat, p_value_w = stats.wilcoxon(base_relato_anon["entrpy_result_orig"], base_relato_anon["entrpy_result_anon"])

In [ ]:
print(f"{w_stat} \n{p_value_w}")

#### Teste T - para calculo de significancia também

In [ ]:
t_stat, p_value_t = stats.ttest_rel(base_relato_anon["entrpy_result_orig"], base_relato_anon["entrpy_result_anon"])

In [ ]:
print(f"{w_stat} \n{p_value_w}")

#### pedi pro chatgpt me ajudar a interpretar ele criou isso aqui KKKKKKKKK

In [ ]:
alpha = 0.05
if p_value_t < alpha:
    print("A diferença entre as entropias é estatisticamente significativa (Teste t).")
else:
    print("A diferença entre as entropias NÃO é estatisticamente significativa (Teste t).")

if p_value_w < alpha:
    print("A diferença entre as entropias é estatisticamente significativa (Teste de Wilcoxon).")
else:
    print("A diferença entre as entropias NÃO é estatisticamente significativa (Teste de Wilcoxon).")

Media dos valores

In [ ]:
base_relato_anon['entrpy_result_orig'].values.mean()

In [ ]:
base_relato_anon['entrpy_result_anon'].values.mean()

In [ ]:
np.std(base_relato_anon['entrpy_result_orig'].values)

In [ ]:
np.std(base_relato_anon['entrpy_result_anon'].values)